# Guardant Health — Hands-On Lab Workbook

Work through this at your own pace. **You cannot break anything** — every write
goes into your own private schema.

### Before you start
You should already have run `lab/02_participant_setup.sql`. If you have not, do
that first: it gives you your own working area.

### How to use this workbook
- Six stations. Each one ends with a **CHECKPOINT** telling you what you should see.
- If a checkpoint does not match, or you fall behind: run the **ESCAPE** cell at
  the end of that station and move on. You will be in the right place for the
  next station.
- Exercises are marked **YOUR TURN** and are one line of code each. The answer is
  in the cell immediately after, so do not scroll if you want to try it.

Put your hand up whenever something does not work. That is what the session is for.

---
## Station 0 — Where am I?

Confirms you are pointed at your own schema and can see the shared data.

In [ ]:
USE ROLE GUARDANT_LAB;
USE WAREHOUSE GUARDANT_LAB_WH;

SET my_schema = 'GUARDANT_LAB.LAB_'
                || UPPER(REGEXP_REPLACE(CURRENT_USER(), '[^A-Za-z0-9]', '_'));
USE SCHEMA IDENTIFIER($my_schema);

SELECT CURRENT_USER() AS you,
       CURRENT_SCHEMA() AS your_private_schema,
       (SELECT COUNT(*) FROM GUARDANT_LAB.GUARDANT_DEMO.VARIANT_CALLS) AS shared_variant_calls;

**CHECKPOINT** — `your_private_schema` contains your username, and
`shared_variant_calls` is **20,000,000**.

If the count fails, you are missing a SELECT grant. Hand up.

---
## Station 1 — Notebooks (~10 min)

The point of this station: **this is your Jupyter, but the compute is in Snowflake.**

Today you query Snowflake, download the result, and process it locally. Watch how
far you get here without downloading anything.

In [ ]:
-- The table you are about to work with.
SELECT COUNT(*)                                         AS raw_variant_calls,
       COUNT(DISTINCT specimen_id)                      AS specimens,
       ROUND(COUNT(*) / COUNT(DISTINCT specimen_id), 1) AS avg_calls_per_specimen
FROM GUARDANT_LAB.GUARDANT_DEMO.VARIANT_CALLS;

In [ ]:
-- The aggregation runs on 20M rows. Only 15 rows come back.
SELECT gene_symbol,
       COUNT(*)                    AS reportable_calls,
       COUNT(DISTINCT specimen_id) AS specimens_affected,
       ROUND(MEDIAN(vaf) * 100, 3) AS median_vaf_pct,
       MAX(is_actionable)          AS has_targeted_therapy
FROM GUARDANT_LAB.GUARDANT_DEMO.V_REPORTABLE_VARIANTS
GROUP BY gene_symbol
ORDER BY reportable_calls DESC
LIMIT 15;

### The handoff

Any SQL cell is available in Python **by its cell name**. No connector, no
cursor, no credentials, no download.

In [ ]:
df = sql_s1_burden.to_pandas()

print(f"Rows now in Python:        {len(df):,}")
print(f"Rows scanned in Snowflake: 20,000,000")
df.head()

### YOUR TURN

Plot it. One line — fill in the column name for the bar lengths:

```python
ax.barh(plot_df["GENE_SYMBOL"], plot_df[" ... "])
```

In [ ]:
import matplotlib.pyplot as plt

plot_df = df.sort_values("REPORTABLE_CALLS")
colors = ["#29B5E8" if a else "#B0BEC5" for a in plot_df["HAS_TARGETED_THERAPY"]]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_df["GENE_SYMBOL"], plot_df["REPORTABLE_CALLS"], color=colors)
ax.set_xlabel("Reportable variant calls")
ax.set_title("Mutation burden by gene (blue = targeted therapy available)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**CHECKPOINT** — you have a horizontal bar chart, TP53 longest, some bars blue.

**ESCAPE** — if the chart did not render, skip it. The table from `sql_s1_burden`
makes the same point and nothing later depends on the plot.

---
## Station 2 — GitHub (~10 min)

The point: **your existing Git workflow, connected — no new process to learn.**

This station is mostly driven in the Snowsight UI, not in this notebook.

1. Open a new browser tab → Snowsight → **Projects → Workspaces**
2. Click **From Git repository**
3. Repository URL: `https://github.com/sfc-gh-pmatson/guardant-snowflake-enablement.git`
4. The repo is **public**, so no credential or token is needed
5. Browse to `notebooks/` — this is the same content you are running now
6. Make an edit, then commit and push from inside Snowsight

**CHECKPOINT** — you can see the repo files in a Workspace and the commit button
is live.

**ESCAPE** — if your account blocks outbound access to github.com, this is a
policy restriction, not a mistake you made. Watch the facilitator's screen and
move to Station 3.

---
## Station 3 — Cortex Code (~10 min)

The point: **an AI assistant that already knows your schema.**

Also driven in the UI. Open Cortex Code in Snowsight and give it this prompt:

> Using GUARDANT_LAB.GUARDANT_DEMO, write me a Snowpark query that finds the
> patients whose KRAS variant allele frequency increased between two blood draws.

Then ask it a follow-up:

> Explain what that query does, line by line.

**Why this matters for your team:** this is how a new joiner gets productive in
days instead of weeks. They can ask what a query does instead of finding the
person who wrote it.

**CHECKPOINT** — you got a query back that references real column names from the
schema, not invented ones.

**ESCAPE** — if Cortex Code is not enabled in your account, watch and move on.

---
## Station 4 — Snowpark (~10 min)

The point: **pandas-style DataFrames, with no memory ceiling.**

The critical idea: a Snowpark DataFrame is **lazy**. It holds a query, not data.
Nothing moves until you ask for a result.

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
from snowflake.snowpark.window import Window

session = get_active_session()

variants  = session.table("GUARDANT_LAB.GUARDANT_DEMO.VARIANT_CALLS")
specimens = session.table("GUARDANT_LAB.GUARDANT_DEMO.SPECIMENS")
panel     = session.table("GUARDANT_LAB.GUARDANT_DEMO.GENE_PANEL")

# Rename the panel's join key so GENE_SYMBOL is never ambiguous downstream.
panel_lookup = panel.select(
    F.col("GENE_SYMBOL").alias("PANEL_GENE"), "IS_ACTIONABLE", "TARGETED_THERAPY"
)

print(f"Rows available: {variants.count():,}")

### Proof it is lazy

Build a four-stage pipeline, then look at what Snowpark actually made. It is SQL.
No rows have moved.

In [ ]:
pipeline = (
    variants
    .filter(F.col("CALL_FILTER") == "PASS")
    .join(specimens, on="SPECIMEN_ID")
    .group_by("GENE_SYMBOL")
    .agg(F.count("*").alias("N"))
)

print(pipeline.queries["queries"][0][:600])

### YOUR TURN

Aggregate across all 20M rows. Fill in the filter so you only count calls that
passed QC:

```python
.filter(F.col(" ... ") == "PASS")
```

In [ ]:
import time

gene_stats = (
    variants
    .filter(F.col("CALL_FILTER") == "PASS")
    .join(panel_lookup, variants["GENE_SYMBOL"] == panel_lookup["PANEL_GENE"])
    .group_by("GENE_SYMBOL", "IS_ACTIONABLE")
    .agg(
        F.count("*").alias("PASS_CALLS"),
        F.count_distinct("SPECIMEN_ID").alias("SPECIMENS"),
        F.round(F.median("VAF") * 100, 3).alias("MEDIAN_VAF_PCT"),
    )
    .sort(F.col("PASS_CALLS").desc())
)

t0 = time.time()
out = gene_stats.to_pandas()
print(f"Aggregated 20M rows in {time.time() - t0:.1f}s, returned {len(out)} rows")
out.head(10)

### The question you actually care about

Not "what is the VAF" but "is it **rising** across serial draws". That is a
window function over each patient's draw history — the operation that gets
painful in pandas, because it needs the whole partition in memory.

In [ ]:
w = Window.partition_by("PATIENT_ID", "GENE_SYMBOL").order_by("COLLECTION_DATE")

trajectory = (
    variants
    .filter((F.col("CALL_FILTER") == "PASS") & (F.col("VAF") >= 0.02))
    .join(specimens, on="SPECIMEN_ID")
    .filter(F.col("QC_STATUS") == "PASS")
    .select("PATIENT_ID", "GENE_SYMBOL", "COLLECTION_DATE", "VAF")
    .with_column("PREV_VAF", F.lag("VAF").over(w))
    .filter(F.col("PREV_VAF").is_not_null())
    .with_column("VAF_DELTA", F.round((F.col("VAF") - F.col("PREV_VAF")) * 100, 3))
)

rising = (
    trajectory
    .group_by("GENE_SYMBOL")
    .agg(
        F.count("*").alias("PAIRED_OBSERVATIONS"),
        F.sum(F.iff(F.col("VAF_DELTA") > 0, 1, 0)).alias("RISING"),
    )
    .with_column("PCT_RISING",
                 F.round(100 * F.col("RISING") / F.col("PAIRED_OBSERVATIONS"), 1))
    .sort(F.col("PAIRED_OBSERVATIONS").desc())
)

rising.to_pandas().head(10)

### Write the result back to your own schema

No extract, no upload. `save_as_table` persists it server-side.

In [ ]:
my_cohort = (
    variants
    .filter(F.col("CALL_FILTER") == "PASS")
    .join(panel_lookup, variants["GENE_SYMBOL"] == panel_lookup["PANEL_GENE"])
    .filter(F.col("IS_ACTIONABLE") & (F.col("VAF") >= 0.05))
    .select("SPECIMEN_ID", "GENE_SYMBOL", "VAF", "READ_DEPTH", "TARGETED_THERAPY")
)

# Unqualified name = your own schema, because Station 0 set the context.
my_cohort.write.mode("overwrite").save_as_table("MY_ACTIONABLE_COHORT")

print(f"MY_ACTIONABLE_COHORT: {session.table('MY_ACTIONABLE_COHORT').count():,} rows")

**CHECKPOINT** — `MY_ACTIONABLE_COHORT` exists in your schema with roughly
3–4 million rows.

**ESCAPE** — run the cell below to create it in one statement and move on.

In [ ]:
-- ESCAPE for Station 4: same end state, one statement.
CREATE OR REPLACE TABLE MY_ACTIONABLE_COHORT AS
SELECT specimen_id, gene_symbol, vaf, read_depth, targeted_therapy
FROM GUARDANT_LAB.GUARDANT_DEMO.V_REPORTABLE_VARIANTS
WHERE is_actionable AND vaf >= 0.05;

SELECT COUNT(*) AS my_cohort_rows FROM MY_ACTIONABLE_COHORT;

---
## Station 5 — Cortex AI (~10 min)

The point: **the unstructured half of your data becomes queryable.**

2,000 synthetic pathology narratives. The facts are in the prose, not in columns,
which is why regex is the wrong tool. And every call below runs inside Snowflake —
no report text goes to an external endpoint.

In [ ]:
-- Read one, so you can see what the model is working with.
SELECT report_text
FROM GUARDANT_LAB.GUARDANT_DEMO.PATHOLOGY_REPORTS
LIMIT 1;

### YOUR TURN

`AI_EXTRACT` turns prose into columns. Add a fourth question of your own to the
`responseFormat` below — anything you would actually want off a report.

In [ ]:
SELECT
    report_id,
    AI_EXTRACT(
        text => report_text,
        responseFormat => {
            'cancer_type': 'What is the primary cancer type?',
            'gene':        'Which gene had the reported alteration?',
            'targetable':  'Is a targeted therapy indicated? Answer only yes or no.'
        }
    ) AS extracted
FROM GUARDANT_LAB.GUARDANT_DEMO.PATHOLOGY_REPORTS
LIMIT 5;

### Triage the whole corpus

`AI_CLASSIFY` routes reports into the buckets a molecular tumour board cares
about. No model to train, categories supplied inline.

In [ ]:
WITH classified AS (
    SELECT AI_CLASSIFY(
             report_text,
             ['actionable alteration found',
              'no actionable alteration',
              'resistance mechanism emerging',
              'no change from prior testing']
           ):labels[0]::VARCHAR AS triage_bucket
    FROM GUARDANT_LAB.GUARDANT_DEMO.PATHOLOGY_REPORTS
    LIMIT 100
)
SELECT triage_bucket,
       COUNT(*)                                           AS reports,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM classified
GROUP BY triage_bucket
ORDER BY reports DESC;

### Reasoning across many rows at once

`AI_AGG` is the one with no laptop equivalent: it reasons over a whole *group* of
text values rather than row by row.

In [ ]:
WITH cohort AS (
    SELECT r.report_text, p.primary_cancer_type
    FROM GUARDANT_LAB.GUARDANT_DEMO.PATHOLOGY_REPORTS r
    JOIN GUARDANT_LAB.GUARDANT_DEMO.SPECIMENS s ON s.specimen_id = r.specimen_id
    JOIN GUARDANT_LAB.GUARDANT_DEMO.PATIENTS  p ON p.patient_id  = s.patient_id
    WHERE p.primary_cancer_type = 'Non-Small Cell Lung'
    LIMIT 60
)
SELECT primary_cancer_type,
       COUNT(*) AS reports_reviewed,
       AI_AGG(report_text,
              'In 3 short bullets: the most common alterations mentioned, any
               recurring resistance patterns, and the dominant recommended next
               step. Be specific and concise.') AS cohort_themes
FROM cohort
GROUP BY primary_cancer_type;

**CHECKPOINT** — the extraction returned a gene name matching the narrative, and
`AI_AGG` produced three bullets.

**ESCAPE** — if a Cortex call errors on model availability, that is a region
setting, not your mistake. Skip to Station 6.

---
## Station 6 — Snowsight (~3 min)

The point: **your analyst colleagues get the same governed data, without code.**

In Snowsight: **Projects → Dashboards → New Dashboard**, add a tile, and paste:

```sql
SELECT primary_cancer_type,
       COUNT(DISTINCT patient_id) AS patients_with_actionable_variant
FROM GUARDANT_LAB.GUARDANT_DEMO.V_REPORTABLE_VARIANTS
WHERE is_actionable AND vaf >= 0.05
GROUP BY primary_cancer_type
ORDER BY patients_with_actionable_variant DESC
LIMIT 12;
```

Switch the tile to **Chart → Bar**. More tile queries are in
`dashboards/snowsight_dashboard_queries.sql`.

**CHECKPOINT** — a bar chart, no CSV involved anywhere.

**The thing to notice:** that dashboard reads the same tables you just used from
Python, under the same permissions. One copy of the data, one set of controls.

---
## What you just did

| | Before | Now |
|---|---|---|
| Where compute ran | Your laptop | Snowflake |
| Data movement | Full extract every time | Results only |
| Environment setup | Per person, per machine | None |
| Ceiling | Local RAM | Warehouse size |
| Unstructured data | Read by hand, or not at all | Queryable |
| Analyst access | You export a CSV for them | They self-serve |

### Take it with you

Everything here is public: **github.com/sfc-gh-pmatson/guardant-snowflake-enablement**

The dataset regenerates from SQL in about 30 seconds, so you can rebuild this
whole lab in any account.

### Your working area

Your schema and everything in it survives the session. If you want it cleaned up,
say so — otherwise it stays for you to keep poking at.